In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)


In [ ]:
# Define the project path, if needed. If None, it will use the the community contributions directory as the project path.
PROJECT_PATH = None
project_dir = os.path.abspath(os.path.join(project_dir, os.pardir))

# Set the model
model = "gpt-4.1-mini"

In [ ]:
params = {"command": "uvx","args": ["code-index-mcp"]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [ ]:
instructions = f"Set the project path to this: {project_dir}"

In [ ]:
# Test the code-index-mcp tool

request = "Find all Python files which has potential security problems in the project and list their paths. Return with a list of file absolute paths only."
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
code_sanitizer_params = {"command": "uvx","args": ["mcp-code-sanitizer"], "env": {"GROQ_API_KEY": os.getenv("GROQ_API_KEY")}}

async with MCPServerStdio(params=code_sanitizer_params, client_session_timeout_seconds=30) as code_sanitizer_server:
    mcp_tools = await code_sanitizer_server.list_tools()

mcp_tools

In [ ]:
code_sanitizer_request = "Analyze code of /home/misi/src/learning/udemy/llm_ed_donner_agents/6_mcp/community_contributions/AdnanGobeljic/agent.py and find potential security problems. If you find any, suggest how to fix them."
async with MCPServerStdio(params=code_sanitizer_params, client_session_timeout_seconds=30) as code_sanitizer_mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[code_sanitizer_mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, code_sanitizer_request)
    display(Markdown(result.final_output))

In [ ]:
import gradio as gr

async def chat(message, history):
    conversation = [
        {"role": item["role"], "content": item["content"]}
        for item in history
    ] + [{"role": "user", "content": message}]
    async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
        agent = Agent(
            name="code_index_chat",
            instructions=instructions,
            model=model,
            mcp_servers=[mcp_server],
        )
        with trace("code_index_chat"):
            result = await Runner.run(agent, conversation)
    return result.final_output

gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Code Index MCP Chat",
    description="Ask questions about the indexed project. The agent uses code-index-mcp tools when needed.",
).launch()